# Figures — El Niño Report (standalone, no need to re-run the main pipeline)

هذا نوتبوك خفيف مستقل: يقرأ فقط الملفات الجاهزة اللي طلعت من نوتبوكك الأصلي (`nino_indices.csv` و `table_modoki_vs_canonical.csv`)، ويرسم:

1. **Figure 1** — خريطة حقيقية لمناطق رصد إنسو (Niño 1+2, 3, 3.4, 4) والمسافة للسعودية، بديل للرسم التخطيطي
2. **خريطة قسم 4.8** — أي نمط (Canonical/Modoki) يهيمن على كل مدينة سعودية

ارفعي ملفي الـCSV هذول لنفس مجلد Colab قبل التشغيل (أو خليهم كما هم لو النوتبوك الأصلي حفظهم أصلاً في نفس الجلسة).

## Setup

In [ ]:
!pip install -q cartopy

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import io


## Figure 1 — Real map of ENSO monitoring regions vs Saudi Arabia

حدود المناطق الأربعة الرسمية (NOAA / Climate Prediction Center):
- Niño 1+2: 0°–10°S, 80°W–90°W
- Niño 3: 5°N–5°S, 90°W–150°W
- Niño 3.4: 5°N–5°S, 120°W–170°W
- Niño 4: 5°N–5°S, 150°W–160°E (يقطع خط التاريخ الدولي، لذا مقسّم لصندوقين في الكود)

المصدر: https://www.noaa.gov/jetstream/tropical/enso

In [ ]:
fig = plt.figure(figsize=(13, 6.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-180, 100, -30, 35], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor='#e8e4d8', zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor='#dbe9f4', zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#777')
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor='#aaa')

# real, standard NOAA box boundaries
NINO_BOXES = {
    'Niño 1+2': {'lon': [(-90, -80)], 'lat': (-10, 0), 'color': '#4A90D9'},
    'Niño 3':   {'lon': [(-150, -90)], 'lat': (-5, 5), 'color': '#27AE60'},
    'Niño 3.4': {'lon': [(-170, -120)], 'lat': (-5, 5), 'color': '#E67E22'},
    'Niño 4':   {'lon': [(150, 180), (-180, -150)], 'lat': (-5, 5), 'color': '#8E44AD'},
}

for name, box in NINO_BOXES.items():
    lat0, lat1 = box['lat']
    for lon0, lon1 in box['lon']:
        rect = mpatches.Rectangle((lon0, lat0), lon1 - lon0, lat1 - lat0,
                                   transform=ccrs.PlateCarree(),
                                   facecolor=box['color'], alpha=0.45,
                                   edgecolor=box['color'], linewidth=1.3, zorder=3)
        ax.add_patch(rect)
    label_lon = box['lon'][0][0] + 5
    ax.text(label_lon, box['lat'][1] + 3, name, transform=ccrs.PlateCarree(),
            fontsize=9, fontweight='bold', color=box['color'], ha='left')

# Riyadh as the Saudi Arabia reference point (same coordinates used throughout the report)
RIYADH = (46.7219, 24.6877)
ax.scatter(*RIYADH, transform=ccrs.PlateCarree(), color='#C0392B', s=110,
           marker='*', zorder=5, edgecolor='white', linewidth=0.8)
ax.text(RIYADH[0] + 3, RIYADH[1] + 2, 'Saudi Arabia', transform=ccrs.PlateCarree(),
        fontsize=10, fontweight='bold', color='#C0392B')

# geodesic (real great-circle) line from Niño 3.4 center to Riyadh
nino34_center = (-145, 0)
ax.plot([nino34_center[0], RIYADH[0]], [nino34_center[1], RIYADH[1]],
        transform=ccrs.Geodetic(), color='#555', linestyle='--', linewidth=1.3, zorder=4)
ax.text(-20, 18, 'teleconnection\n~13,000 km', transform=ccrs.PlateCarree(),
        fontsize=9, style='italic', color='#555', ha='center')

ax.set_title('Where ENSO is measured, and how far its influence must travel',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig1_enso_regions_real.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()


## Section 4.8 — Which El Niño type dominates rainfall response, by city

يقرأ `table_modoki_vs_canonical.csv` (المُنتَج من خلية 9.8e في النوتبوك الأصلي). لو الملف مو موجود بنفس المجلد، الكود يستخدم القيم المحفوظة نفسها كنسخة احتياطية (نفس الأرقام تمامًا اللي طلعت عندك).

In [ ]:
FALLBACK_CSV = """city,La Nina,Canonical,Modoki,Canonical_ratio,Modoki_ratio
Dammam,0.0565,0.0779,0.1150,1.3773,2.0343
Jeddah,0.0606,0.1309,0.0986,2.1596,1.6263
Madinah,0.0414,0.0508,0.0563,1.2272,1.3612
Makkah,0.0997,0.2020,0.1573,2.0268,1.5778
Riyadh,0.0510,0.0666,0.0845,1.3056,1.6568
"""

try:
    df = pd.read_csv('table_modoki_vs_canonical.csv')
    print('loaded table_modoki_vs_canonical.csv from disk')
except FileNotFoundError:
    df = pd.read_csv(io.StringIO(FALLBACK_CSV))
    print('table_modoki_vs_canonical.csv not found — using saved values from the original run')

COORDS = {
    'Riyadh': (24.6877, 46.7219), 'Jeddah': (21.4858, 39.1925),
    'Makkah': (21.4225, 39.8262), 'Madinah': (24.4686, 39.6142),
    'Dammam': (26.4207, 50.0888),
}
df['lat'] = df['city'].map(lambda c: COORDS[c][0])
df['lon'] = df['city'].map(lambda c: COORDS[c][1])
df['dominant'] = df.apply(lambda r: 'Canonical' if r['Canonical_ratio'] > r['Modoki_ratio'] else 'Modoki', axis=1)
df['dominant_ratio'] = df[['Canonical_ratio', 'Modoki_ratio']].max(axis=1)
print(df[['city', 'Canonical_ratio', 'Modoki_ratio', 'dominant']])


In [ ]:
fig = plt.figure(figsize=(8, 7.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([34, 52, 15, 33], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor='#f5f0e6', zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor='#dbe9f4', zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor='#777')
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='#aaa')
ax.gridlines(draw_labels=True, linewidth=0.3, color='#ccc')

colors = {'Canonical': '#8E44AD', 'Modoki': '#E67E22'}
for _, r in df.iterrows():
    size = 350 * r['dominant_ratio'] / df['dominant_ratio'].max()
    ax.scatter(r['lon'], r['lat'], transform=ccrs.PlateCarree(),
               s=size, color=colors[r['dominant']], edgecolor='white',
               linewidth=1.2, zorder=5)
    ax.text(r['lon'] + 0.35, r['lat'], f"{r['city']}\n{r['dominant']} {r['dominant_ratio']:.2f}x",
            transform=ccrs.PlateCarree(), fontsize=9, zorder=6,
            fontweight='bold', color=colors[r['dominant']])

handles = [plt.Line2D([0], [0], marker='o', color='w', label=k,
           markerfacecolor=v, markersize=12) for k, v in colors.items()]
ax.legend(handles=handles, loc='lower right', frameon=True, fontsize=9)

ax.set_title('Which El Niño Type Dominates Rainfall Response, by City',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig_modoki_map.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
